# Leaves in the joint model: reconstruct only what can be predicted
**The problem.** In the joint arm (reconstruction of every frame + prediction through the encoder) the unpredictable content (the "leaves": here the four
twinkle cells, and the mob's random steps) is not removed. The decoder is a consumer with free gain, so the encoder can keep the noise at **low amplitude**:
its cost in the prediction loss scales with its amplitude, while the decoder multiplies it back up. (Observed in the earlier runs: twinkle read at 0.83-0.93
by the model's own decoder after 40k steps, predicted at -1.1; the latent barely changed from frame to frame, copy-last error 0.01.)

**The new term: decoded prediction** (`Cfg.dp_weight`). The predictor's output is decoded and compared with the **real next frame**, with the same class-weighted
pixel error as the reconstruction: `||D(P(z_ctx, a)) - x_next||`. A prediction cannot contain unpredictable content, so this term asks for nothing but what
can be predicted. (Earlier, on a frozen shadow, it lifted the agent by +0.20 and the facing by +0.23 on move steps.)

| idea | what | runs |
|---|---|---|
| baseline | reconstruction of every frame + latent prediction (the joint arm as before) | **joint** |
| **idea 2**: reconstruct only what can be predicted | the content-agnostic reconstruction is **removed**; the decoded prediction is the only reconstruction-like term | **dp only** |
| **idea 3**: decoded prediction as the consolidation | warm up with reconstruction + decoded prediction; later **fade only the reconstruction**, the decoded prediction stays (it holds what is predicted, ignores what is not, and unlike a snapshot anchor it keeps improving because its target is real data) | **joint + dp**, then forks |

Same weights everywhere (ground-truth tile classes, cap 4, FiLM predictor, D = 128, 20k steps, seed 0). Every claim is read with decoders
(`gaf/readout.py`), plus the latent's **variance shares** (how loud each part is in the latent) and a timeline over the checkpoints.

**Pre-registered expectations** (written before the results; the verdict cells apply them):

*Idea 2 (`dp only`)*
* **H2a, leaves not demanded:** twinkle in the latent is much weaker than in `joint`: at least two of three measures agree (the model's own decoder on the real next
  frame drops by >= 0.3; the variance share of twinkle falls to <= half; a fresh decoder drops by >= 0.2).
* **H2b, dynamics content kept:** agent, hand, facing and inventory are held and predicted as well as in `joint`: predicted move-step facing within -0.05 of `joint`,
  predicted hand at least as high.
* **H2c, the cost (a risk, not an expectation):** content that is not demanded by prediction early is never made available: the mob and the terrain in the real
  next frame are lower than in `joint`, and the timeline shows later learning.

*Idea 3 (`joint + dp`, then forks)*
* **H3a, the term helps the dynamics in the warmup:** predicted move-step facing or hand at least 0.05 above `joint`.
* **H3b, consolidation:** `fade` from `joint + dp` **holds** the learned content (no drop > 0.05 in agent, inventory, facing, by the fresh decoder and the own decoder),
  **releases the noise** (two of the three measures above), and keeps the dynamics (predicted move-step facing no more than 0.05 below the parent).
* **H3c, control:** `fade` from plain `joint` erodes (facing drops by >= 0.2 by the fresh decoder), as it did before.
* The stricter `fade` to a reconstruction weight of **0** is run as a variant.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from gaf.env_grid import GridCfg
from gaf.train import Cfg
from gaf import analysis as A
from gaf import readout as R
from gaf import fork as F

OUT = "/workspace/results_joint_dp"
SEED, D, STEPS, WORKERS = 0, 128, 20_000, 3
FORK_STEPS, FADE = 12_000, 4_000          # stage 2: 4000 steps of fade, then 8000 at the floor
DS_STEPS = 5000                           # steps of the fresh decoder of the decoder-strength readout (the same for every row)
TC_STEPS = (5000, 10000, 15000, 20000)    # checkpoints of the timeline
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=STEPS, floor_start=10 ** 9, eval_every=2000, eval_parts=6000, eval_parts_steps=500,
              D=D, seed=SEED, env=ENV, rec_class_power=1.0, rec_class_cap=4.0, pred_arch="film", ckpt_every=5000)
RUNS = {"joint": Cfg(arm="joint", tag="dp-joint", **COMMON),
        "joint + dp": Cfg(arm="joint", tag="dp-joint-dp", dp_weight=1.0, **COMMON),
        "dp only": Cfg(arm="joint", tag="dp-only", dp_weight=1.0, shadow_rec=0.0, **COMMON)}
DIR = {n: Path(OUT) / c.run_id for n, c in RUNS.items()}
P_DP, P_J = DIR["joint + dp"], DIR["joint"]                 # the parents of the forks
COLOR = dict(zip(RUNS, A.RUN_COLORS))
pd.set_option("display.precision", 3)

## Stage 1: launch
Three runs from scratch in a background process (safe to re-run: finished and running runs are skipped). To stop: `kill -TERM -- -$(cat /workspace/results_joint_dp/launch_*.pid)`.

In [ ]:
A.launch(list(RUNS.values()), OUT, workers=WORKERS)

In [ ]:
display(A.progress(OUT, list(RUNS.values())))

In [ ]:
A.live_probes(OUT, RUNS, every=30)

**Recovery cell.** Only if a run died in its final probes (it has `ckpt_<steps>.pt` and a full `metrics.csv` but no `summary.json`).

In [ ]:
for name, c in RUNS.items():
    d = DIR[name]
    if d.exists() and not (d / "summary.json").exists() and (d / f"ckpt_{c.steps}.pt").exists():
        print("finalizing", name)
        A.finalize(c, OUT)

## Stage 1, analysis 1: what each model foresees at 20k
Rows per run: the model's own decoder applied to the **real next latent** (what the latent holds), to the **predictor's output** (what it foresees) and to the
**last frame** (the floor). Columns are decoded tile classes (1 = perfect, 0 = nothing, negative = worse than "decode nothing") and the facing orientation accuracy
(chance 0.25; "move steps" are the clips where the last action moved or turned the agent, where copying is mostly wrong).

In [ ]:
dyn1 = {n: R.dynamics(DIR[n]) for n in RUNS if (DIR[n] / "summary.json").exists()}
DYN1 = pd.concat({n: d.loc["live"] for n, d in dyn1.items()})
display(DYN1.round(2))

## Stage 1, analysis 2: when is it learned? The timeline over the checkpoints
The same readouts at 5k, 10k, 15k and 20k steps. Left: what the model **foresees** (decoded predicted frame, move-step facing, hand, agent, mob). Right: what the
latent **holds** (real next frame, own decoder: mob, twinkle, terrain) and how **loud** the twinkle is in the latent (variance share).

In [ ]:
def timecourse(d):
    rows = {}
    for st in TC_STEPS:
        dyn = R.dynamics(d, n_eval=1024, step=st).loc["live"]
        vs = R.variance_shares(d, n=2048, step=st)
        rows[st] = {"predicted facing (move)": dyn.loc["predicted", "facing acc, move steps"], "predicted hand": dyn.loc["predicted", "hand"],
                    "predicted agent": dyn.loc["predicted", "agent"], "predicted mob": dyn.loc["predicted", "mob"],
                    "latent holds: mob": dyn.loc["actual next frame", "mob"], "latent holds: terrain": dyn.loc["actual next frame", "map"],
                    "latent holds: twinkle": dyn.loc["actual next frame", "twinkle"], "twinkle variance share": vs["twinkle"]}
    return pd.DataFrame(rows).T


TC = {n: timecourse(DIR[n]) for n in RUNS if (DIR[n] / "summary.json").exists()}
cols = list(next(iter(TC.values())).columns)
fig, axes = plt.subplots(2, 4, figsize=(18, 7), layout="constrained")
for ax, col in zip(axes.flat, cols):
    for n, df in TC.items():
        ax.plot(df.index, df[col], color=COLOR[n], marker="o", label=n)
    ax.set(title=col, xlabel="step")
axes.flat[0].legend(fontsize=8)
plt.show()
display(pd.concat(TC).round(3))

## Stage 1, analysis 3: the leaves
Three independent instruments for "is the noise in the latent?": the model's own decoder on the real next latent, the **variance share** of twinkle in the latent
(a low value means it sits at low amplitude), and a **fresh decoder** trained on the frozen latent. Plus the final MLP probe of the twinkle values and the latent's
effective rank. Reference from the earlier joint run (40k steps): own decoder 0.83-0.93, fresh decoder 0.31-0.38.

In [ ]:
content1 = {n: R.decoder_strength(DIR[n], steps=DS_STEPS).loc["live"] for n in dyn1}
CONTENT1 = pd.DataFrame(content1).T
leaf = {}
for n in dyn1:
    d = dyn1[n].loc["live"]
    summ = A.collect(OUT)
    row = summ[summ.run_id == RUNS[n].run_id]
    leaf[n] = {"own decoder: real next frame": d.loc["actual next frame", "twinkle"], "own decoder: predicted": d.loc["predicted", "twinkle"],
               "copy last": d.loc["copy last frame", "twinkle"], "variance share": R.variance_shares(DIR[n])["twinkle"],
               "fresh decoder": CONTENT1.loc[n, "twinkle"], "MLP probe (final)": float(row["pm_twinkle"].iloc[0]) if len(row) else np.nan,
               "latent effective rank": float(row["pm_rank"].iloc[0]) if len(row) else np.nan}
LEAF = pd.DataFrame(leaf).T
print("decoder strength (fresh decoder) of every part:")
display(CONTENT1.round(2))
print("the twinkle (the leaves), five instruments:")
display(LEAF.round(3))

## Stage 1: verdicts (rules written before the results)

In [ ]:
def g(df, idx, col):
    try:
        return float(df.loc[idx, col])
    except (KeyError, TypeError):
        return float("nan")


def say(label, ok, detail=""):
    print(f"{label}: {'yes' if ok else 'NO'}  {detail}")


if len(dyn1) == 3:
    j, jd, do = DYN1.loc["joint"], DYN1.loc["joint + dp"], DYN1.loc["dp only"]
    # H2a: leaves not demanded in dp only
    votes = [g(DYN1, ("joint", "actual next frame"), "twinkle") - g(DYN1, ("dp only", "actual next frame"), "twinkle") >= 0.3,
             LEAF.loc["dp only", "variance share"] <= 0.5 * LEAF.loc["joint", "variance share"],
             CONTENT1.loc["joint", "twinkle"] - CONTENT1.loc["dp only", "twinkle"] >= 0.2]
    say("H2a leaves not demanded in `dp only` (>= 2 of 3 instruments)", sum(votes) >= 2, f"votes {votes}")
    # H2b: dynamics content kept
    pf = lambda n: g(DYN1, (n, "predicted"), "facing acc, move steps")
    ph = lambda n: g(DYN1, (n, "predicted"), "hand")
    say("H2b dynamics content kept in `dp only`", pf("dp only") >= pf("joint") - 0.05 and ph("dp only") >= ph("joint"),
        f"predicted move-step facing {pf('dp only'):.2f} vs {pf('joint'):.2f}; predicted hand {ph('dp only'):.2f} vs {ph('joint'):.2f}")
    # H2c: the cost
    cost = {k: (g(DYN1, ("joint", "actual next frame"), k), g(DYN1, ("dp only", "actual next frame"), k)) for k in ("mob", "map")}
    say("H2c the cost: mob/terrain lower in `dp only`", all(b < a - 0.1 for a, b in cost.values()), f"(joint, dp only): {cost}")
    # H3a
    say("H3a the decoded prediction helps in the warmup (`joint + dp` vs `joint`)", pf("joint + dp") >= pf("joint") + 0.05 or ph("joint + dp") >= ph("joint") + 0.05,
        f"move-step facing {pf('joint + dp'):.2f} vs {pf('joint'):.2f}; hand {ph('joint + dp'):.2f} vs {ph('joint'):.2f}")
else:
    print("stage 1 not finished")

## Stage 2: the fade, with the decoded prediction kept (idea 3)
Forks of the 20k checkpoints (no retraining of the warmup), 12,000 steps each, 4000 of them fading the **content-agnostic reconstruction** (the decoded prediction stays
at full weight in every fork of `joint + dp`):

| fork | parent | reconstruction weight |
|---|---|---|
| `continue` | joint + dp | stays 1 (the control) |
| `fade` | joint + dp | 1 to 0.1 |
| `fade-r0` | joint + dp | 1 to 0 (the strict version) |
| `fade` | joint | 1 to 0.1, **no** decoded prediction: the control that eroded before |

Start this after the Stage 1 verdict (or anyway: the forks are cheap).

In [ ]:
print(F.launch_forks(P_DP, modes=("continue", "fade"), steps=FORK_STEPS, fade=FADE, r_floor=0.1))
print(F.launch_forks(P_DP, modes=("fade",), steps=FORK_STEPS, fade=FADE, r_floor=0.0, tag="r0"))
print(F.launch_forks(P_J, modes=("fade",), steps=FORK_STEPS, fade=FADE, r_floor=0.1))

In [ ]:
for parent, names in ((P_DP, ["continue", "fade", "fade-r0"]), (P_J, ["fade"])):
    print(parent.name)
    display(F.status(parent, modes=names))

## Stage 2, analysis: what the fade holds, releases and costs
For every fork and its parent: the fresh-decoder content, the own decoder on the real next latent, the decoded predicted frame, and the three measures of the noise.
**holds** = no drop > 0.05 in agent, inventory, facing (fresh decoder) and in the own-decoder agent and facing; **releases noise** = two of three measures
(own decoder drop >= 0.3; variance share <= half of the parent's; fresh decoder drop >= 0.2); **dynamics kept** = predicted move-step facing no more than 0.05 below the parent.

In [ ]:
FORKS = {"joint + dp: continue": (P_DP, P_DP / "forks" / "continue", "joint + dp"), "joint + dp: fade (0.1)": (P_DP, P_DP / "forks" / "fade", "joint + dp"),
         "joint + dp: fade (0)": (P_DP, P_DP / "forks" / "fade-r0", "joint + dp"), "joint: fade (0.1)": (P_J, P_J / "forks" / "fade", "joint")}
done = {n: p for n, (_, p, _) in FORKS.items() if (p / "done").exists()}
s2 = {}
for n, p in done.items():
    dd = R.dynamics(p).loc["live"]
    s2[n] = {"dyn": dd, "content": R.decoder_strength(p, steps=DS_STEPS).loc["live"], "vs": R.variance_shares(p)["twinkle"]}
rows = {}
for n, v in s2.items():
    rows[n] = {
        "fresh: agent": v["content"]["agent"], "fresh: hand": v["content"]["hand"], "fresh: inventory": v["content"]["inventory"], "fresh: facing": v["content"]["facing acc"],
        "own: agent": v["dyn"].loc["actual next frame", "agent"], "own: facing": v["dyn"].loc["actual next frame", "facing acc"], "own: mob": v["dyn"].loc["actual next frame", "mob"],
        "pred: facing (move)": v["dyn"].loc["predicted", "facing acc, move steps"], "pred: hand": v["dyn"].loc["predicted", "hand"], "pred: agent": v["dyn"].loc["predicted", "agent"],
        "twinkle: own decoder": v["dyn"].loc["actual next frame", "twinkle"], "twinkle: variance share": v["vs"], "twinkle: fresh": v["content"]["twinkle"]}
S2 = pd.DataFrame(rows).T
parents = pd.DataFrame({
    n: {"fresh: agent": CONTENT1.loc[n, "agent"], "fresh: hand": CONTENT1.loc[n, "hand"], "fresh: inventory": CONTENT1.loc[n, "inventory"], "fresh: facing": CONTENT1.loc[n, "facing acc"],
        "own: agent": DYN1.loc[(n, "actual next frame"), "agent"], "own: facing": DYN1.loc[(n, "actual next frame"), "facing acc"], "own: mob": DYN1.loc[(n, "actual next frame"), "mob"],
        "pred: facing (move)": DYN1.loc[(n, "predicted"), "facing acc, move steps"], "pred: hand": DYN1.loc[(n, "predicted"), "hand"], "pred: agent": DYN1.loc[(n, "predicted"), "agent"],
        "twinkle: own decoder": DYN1.loc[(n, "actual next frame"), "twinkle"], "twinkle: variance share": LEAF.loc[n, "variance share"], "twinkle: fresh": CONTENT1.loc[n, "twinkle"]}
    for n in ("joint", "joint + dp")}).T
display(pd.concat([parents, S2]).round(2))

In [ ]:
for n in S2.index:
    par = FORKS[n][2]
    b, f = parents.loc[par], S2.loc[n]
    drop = {k: round(b[k] - f[k], 2) for k in ("fresh: agent", "fresh: inventory", "fresh: facing", "own: agent", "own: facing")}
    holds = all(v <= 0.05 for v in drop.values())
    votes = [b["twinkle: own decoder"] - f["twinkle: own decoder"] >= 0.3, f["twinkle: variance share"] <= 0.5 * b["twinkle: variance share"],
             b["twinkle: fresh"] - f["twinkle: fresh"] >= 0.2]
    dyn_drop = b["pred: facing (move)"] - f["pred: facing (move)"]
    print(f"--- {n}  (parent: {par})")
    print("  drops of learned content:", drop)
    say("  holds", holds)
    say("  releases noise (>= 2 of 3)", sum(votes) >= 2, f"votes {votes}")
    say("  dynamics kept", dyn_drop <= 0.05, f"(predicted move-step facing {f['pred: facing (move)']:.2f} vs parent {b['pred: facing (move)']:.2f})")

## Verdict on the three ideas
Read the printed verdicts together:
* **H3c** (`joint: fade`): should show it erodes and does not release cleanly, otherwise the control did not reproduce.
* **H3b** (`joint + dp: fade`): consolidation works if it holds, releases the noise and keeps the dynamics. Then compare with `joint + dp: continue`: a fade that
  releases noise at no cost is a gain only if noise matters for something; if it only matches `continue`, the simpler model (no fade) is the one to keep.
* **`dp only`** (H2a/H2b/H2c): the simplest model of all (no fade at all). If it removes the leaves and keeps the dynamics content, the price is on the availability side (H2c).

The table below puts every model on one line for the decision.

In [ ]:
final = {}
for n in RUNS:
    if n in dyn1:
        d = dyn1[n].loc["live"]
        final[n] = {"pred facing (move)": d.loc["predicted", "facing acc, move steps"], "pred hand": d.loc["predicted", "hand"], "pred agent": d.loc["predicted", "agent"],
                    "holds: hand (fresh)": CONTENT1.loc[n, "hand"], "holds: facing (fresh)": CONTENT1.loc[n, "facing acc"], "holds: mob (own)": d.loc["actual next frame", "mob"],
                    "twinkle: own": d.loc["actual next frame", "twinkle"], "twinkle: variance share": LEAF.loc[n, "variance share"]}
for n, row in S2.iterrows():
    final[n] = {"pred facing (move)": row["pred: facing (move)"], "pred hand": row["pred: hand"], "pred agent": row["pred: agent"], "holds: hand (fresh)": row["fresh: hand"],
                "holds: facing (fresh)": row["fresh: facing"], "holds: mob (own)": row["own: mob"], "twinkle: own": row["twinkle: own decoder"],
                "twinkle: variance share": row["twinkle: variance share"]}
display(pd.DataFrame(final).T.round(2))

## Download
Zip `/workspace/results_joint_dp` (checkpoints and `forks/` included) and download it. It is all you need to redo any readout, or to read an earlier step.